# Анализ и чистка данных

In [1]:
import os
import pandas as pd
import json
import numpy as np

from solution_utils import (
    compute_metrics, compute_partition_metrics, init_reaction_models, init_topic_models,
    predict_reaction, predict_topic, read_dialogs, read_labels, save_predictions,
)

In [47]:
pd.set_option("display.max_colwidth", 150) # максимално символов в ячейке колонки
pd.set_option("display.max_columns", None) # показывать все колонки

In [49]:
DATA_PATH = "hw1_train.jsonl"

records = []
with open(DATA_PATH, encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            record = json.loads(line)
            records.append(record)

records_df = pd.DataFrame(records)
records_df = records_df.set_index("dialog_id")
records_df.head()


,language,n_user_turns,turns,labels
dialog_id,,,,
d094012177e,en,7,"[{'role': 'user', 'text': 'it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and in...","{'task_type': 'analysis', 'reaction': 'positive', 'user_turns': [{'task_type': 'analysis', 'state': 'continuation', 'reaction': None, 'reaction_re..."
d306fbe61fe,en,8,"[{'role': 'user', 'text': 'Can you help me train a small language model that can answer questions, i don't want to use a pretrained mode, i want t...","{'task_type': 'information_seeking', 'reaction': 'neutral', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': N..."
dd6905952c7,en,5,"[{'role': 'user', 'text': 'ask questions about a fictional voice actor i made and based on the answers describe him.'}, {'role': 'assistant', 'tex...","{'task_type': 'creation', 'reaction': 'neutral', 'user_turns': [{'task_type': 'creation', 'state': 'newtopic', 'reaction': None, 'reaction_reason'..."
d767a675896,en,6,"[{'role': 'user', 'text': 'Calculate each prompt based off of data and show work: Calculate: Species Richness Shannon wiener index Species Evene...","{'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ..."
d7a6280ce8e,en,8,"[{'role': 'user', 'text': 'mass spectrometer, the length of AC rod will it affect to intensity? longer or shorter better?'}, {'role': 'assistant',...","{'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ..."


In [53]:
# records = np.array(records)
#records_dict = records_df.to_dict(orient="list")

print(f"всего диалогов: {len(records)}")
print("сколько ПРОПУСКОВ в каждом столбце:")
print(records_df.isna().sum())
ids_set = set(records_df.index)
print(f"всего уникальных dialog_id: {len(ids_set)}")

всего диалогов: 10000
сколько ПРОПУСКОВ в каждом столбце:
language           0
n_user_turns       0
turns              0
labels          7000
dtype: int64
всего уникальных dialog_id: 10000


Выводы: пропуски есть только в labels, размечено всего 3000 диалогов из 10000.
Все id диалогов уникальные, повторов нет

In [54]:
print("ПРИМЕРЫ labels:")
print(records_df["labels"].head(5))

ПРИМЕРЫ labels:
dialog_id
d094012177e    {'task_type': 'analysis', 'reaction': 'positive', 'user_turns': [{'task_type': 'analysis', 'state': 'continuation', 'reaction': None, 'reaction_re...
d306fbe61fe    {'task_type': 'information_seeking', 'reaction': 'neutral', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': N...
dd6905952c7    {'task_type': 'creation', 'reaction': 'neutral', 'user_turns': [{'task_type': 'creation', 'state': 'newtopic', 'reaction': None, 'reaction_reason'...
d767a675896    {'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ...
d7a6280ce8e    {'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ...
Name: labels, dtype: object


In [55]:
print("ПРИМЕРЫ диалогов:")
print(records_df["turns"].head(5))

ПРИМЕРЫ диалогов:
dialog_id
d094012177e    [{'role': 'user', 'text': 'it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and in...
d306fbe61fe    [{'role': 'user', 'text': 'Can you help me train a small language model that can answer questions, i don't want to use a pretrained mode, i want t...
dd6905952c7    [{'role': 'user', 'text': 'ask questions about a fictional voice actor i made and based on the answers describe him.'}, {'role': 'assistant', 'tex...
d767a675896    [{'role': 'user', 'text': 'Calculate each prompt based off of data and show work:

Calculate: 
Species Richness
Shannon wiener index
Species Evene...
d7a6280ce8e    [{'role': 'user', 'text': 'mass spectrometer, the length of AC rod will it affect to intensity? longer or shorter better?'}, {'role': 'assistant',...
Name: turns, dtype: object
